# Project Bài 10 · Tự viết một mạng nơ-ron cho ảnh thời trang

**Học máy cơ bản · Bài 10 · PGS.TS Nguyễn Văn Hậu**

**Tình huống.** Cửa hàng thời trang của project Bài 9 muốn tự động gán loại cho ảnh sản phẩm mới. Bạn viết **từ đầu bằng numpy** một mạng 784 → H → 10 (ReLU, softmax), tự cài lan truyền ngược, kiểm nó bằng sai phân, huấn luyện bằng mini-batch, chọn cấu hình bằng validation, rồi so với logistic trên một tập kiểm tra cuối chấm **một lần**.

**Dữ liệu.** `data/project_subset_bai10.npz`: 10.000 ảnh Fashion-MNIST (Zalando Research, giấy phép MIT) lấy từ 35.000 ảnh train chính thức **không dùng** ở bài giảng Bài 9–10 lẫn project Bài 9 (seed 2027): 8.000 ảnh fit, 1.000 validation, 1.000 kiểm tra cuối; mỗi loại đúng một phần mười. Test chính thức đã dùng ở bài giảng, nên project có tập kiểm tra riêng.

| Mốc | Bước | Ô tự kiểm |
|---|---|---|
| M1 | 1: nạp, kiểm số ảnh | 8.000 / 1.000; mỗi loại 10% |
| M2 | 2: forward, cross-entropy, lan truyền ngược; kiểm sai phân | loss ban đầu gần ln 10; sai lệch tương đối < 10⁻⁷ |
| M3 | 3: huấn luyện mini-batch, checkpoint | loss train giảm; checkpoint đúng epoch CE validation nhỏ nhất |
| M4 | 4 (khá): chọn H ∈ {16; 64} theo quy tắc ghi trước; logistic đối chứng | quy tắc |
| M5 | 5 (khá): chấm kiểm tra cuối **một lần**, so cặp (McNemar) | hàm chặn chấm lần hai |

**Ba mức.** *Cơ bản (5 điểm)*: bước 1–3. *Khá (+3)*: bước 4–5 và báo cáo một trang. *Nâng cao (+2)*: A (đơn vị ẩn so với thành phần PCA) hoặc B (ảnh dịch 1–2 pixel; thêm ảnh dịch khi huấn luyện).

**Luật chơi.** Phần chính không dùng `MLPClassifier`; được dùng để đối chiếu nếu muốn. Mọi lựa chọn làm trên validation. Tập kiểm tra cuối chỉ chấm qua hàm `cham_cuoi` (đã viết sẵn, không sửa).

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import json, time, hashlib
import numpy as np
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd()
assert (ROOT / 'data' / 'project_subset_bai10.npz').exists(), 'Mở notebook từ thư mục bài (có thư mục data/).'
with np.load(ROOT / 'data' / 'project_subset_bai10.npz') as d:
    X = d['fit_images'] / 255.0; y = d['fit_labels'].astype(int)
    V = d['val_images'] / 255.0; yv = d['val_labels'].astype(int)
    _F, _yF = d['final_images'] / 255.0, d['final_labels'].astype(int)      # chỉ dùng qua cham_cuoi

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

_DAU_VET = ROOT / '.project_bai10_cham_cuoi.json'
def cham_cuoi(cau_hinh, du_doan_mlp, du_doan_logistic):
    # Chấm tập kiểm tra cuối đúng MỘT lần. du_doan_* là hàm nhận ma trận ảnh, trả nhãn dự đoán.
    # Khoá gồm chuỗi cấu hình và "dấu vân tay" của hai mô hình (mã băm dự đoán trên VALIDATION, không đụng tập cuối).
    # Gọi lại với đúng cấu hình và đúng mô hình thì nhận lại kết quả cũ (để chạy lại notebook); khác một trong hai sẽ bị chặn.
    # Đây là cơ chế nhắc kỷ luật, không phải bảo mật: giảng viên đối chiếu tệp dấu vết khi chấm.
    van_tay = hashlib.sha256(np.asarray(du_doan_mlp(V), np.int64).tobytes() + np.asarray(du_doan_logistic(V), np.int64).tobytes()).hexdigest()[:16]
    if _DAU_VET.exists():                          # kiểm khoá trước khi tính bất cứ gì trên tập cuối
        cu = json.loads(_DAU_VET.read_text())
        if cu['cau_hinh'] != cau_hinh or cu.get('van_tay') != van_tay:
            raise RuntimeError(f"Tập kiểm tra cuối đã được chấm với {cu['cau_hinh']} (dấu vân tay {cu.get('van_tay')}) lúc {cu['luc']}. "
                               'Không được chấm lại với cấu hình hoặc mô hình khác.')
        return cu['ket_qua']
    pm, pl = du_doan_mlp(_F) == _yF, du_doan_logistic(_F) == _yF
    b, c = int((pm & ~pl).sum()), int((~pm & pl).sum())
    kq = dict(acc_mlp=float(pm.mean()), acc_logistic=float(pl.mean()), chi_mlp_dung=b, chi_logistic_dung=c,
              p_mcnemar=float(binomtest(min(b, c), b + c, 0.5).pvalue) if b + c else 1.0)
    _DAU_VET.write_text(json.dumps(dict(cau_hinh=cau_hinh, van_tay=van_tay, ket_qua=kq, luc=time.strftime('%Y-%m-%d %H:%M'))))
    return kq
print(X.shape, V.shape, _F.shape)

## Bước 1 · Kiểm dữ liệu

Đếm số ảnh mỗi tập và mỗi loại (10 loại, nhãn 0…9).

In [ ]:
# TODO: dem = {'fit': số ảnh mỗi loại của y, 'val': số ảnh mỗi loại của yv}  (gợi ý: np.bincount)
raise NotImplementedError('TODO bước 1: dem')
print(dem)

In [ ]:
# TỰ KIỂM · bước 1
kiem('8.000 ảnh fit, 1.000 ảnh validation', (len(X), len(V)) == (8000, 1000))
kiem('mỗi loại đúng 10%', all((v == v[0]).all() for v in dem.values()))

## Bước 2 · Forward, cross-entropy, lan truyền ngược

Tham số là một dict `P` với `W1` (784 × H), `b1` (H), `W2` (H × 10), `b2` (10). Viết:
- `khoi_tao(d, H, K=10, seed=0)`: khởi tạo ngẫu nhiên nhỏ (gợi ý: đều trong ±√(6/(số vào + số ra)), như scikit-learn);
- `forward(P, X)`: trả `cache` gồm `X`, `A = XW1 + b1`, `H = ReLU(A)`, `S = HW2 + b2`;
- `cross_entropy(S, y)`: trung bình −ln softmax(S)[y], ổn định số (trừ max trước khi lấy mũ);
- `backward(P, cache, y, alpha)`: gradient của `cross_entropy + (alpha/2n)(‖W1‖² + ‖W2‖²)` theo cả bốn tham số (slide 41, 43–44).

In [ ]:
def khoi_tao(d, H, K=10, seed=0):
    raise NotImplementedError('TODO bước 2: khoi_tao')
def forward(P, X):
    raise NotImplementedError('TODO bước 2: forward')
def cross_entropy(S, y):
    raise NotImplementedError('TODO bước 2: cross_entropy')
def backward(P, cache, y, alpha=0.0):
    raise NotImplementedError('TODO bước 2: backward')
P0 = khoi_tao(784, 64, seed=0)
L0 = cross_entropy(forward(P0, X)['S'], y)
print('loss lúc khởi tạo =', round(L0, 4), '· ln 10 =', round(np.log(10), 4))

In [ ]:
# Kiểm gradient bằng sai phân (chạy sẵn, không cần sửa)
# kiểm gradient bằng sai phân trên một mạng nhỏ: 6 đầu vào, 5 đơn vị ẩn, 10 lớp, 8 quan sát
rg = np.random.default_rng(1)
Xs = rg.normal(size=(8, 6)); ys = rg.integers(0, 10, 8)
Ps = khoi_tao(6, 5, seed=3)
gs = backward(Ps, forward(Ps, Xs), ys, alpha=0.5)
def loss_s(P):
    return cross_entropy(forward(P, Xs)['S'], ys) + 0.5 * 0.5 * ((P['W1'] ** 2).sum() + (P['W2'] ** 2).sum()) / len(ys)
sai_lech = 0.0
for ten in ['W1', 'b1', 'W2', 'b2']:
    for idx in np.ndindex(Ps[ten].shape):
        Pp = {k: v.copy() for k, v in Ps.items()}; Pm = {k: v.copy() for k, v in Ps.items()}
        Pp[ten][idx] += 1e-5; Pm[ten][idx] -= 1e-5
        so = (loss_s(Pp) - loss_s(Pm)) / 2e-5
        sai_lech = max(sai_lech, abs(so - gs[ten][idx]) / max(1e-3, abs(so) + abs(gs[ten][idx])))
print('sai lệch tương đối lớn nhất:', sai_lech)

In [ ]:
# TỰ KIỂM · bước 2
kiem('loss lúc khởi tạo gần ln 10 (mô hình chưa học gần như đoán đều)', abs(L0 - np.log(10)) < 0.5, 'khởi tạo có quá lớn không? softmax có trừ max không?')
kiem('lan truyền ngược khớp sai phân (sai lệch tương đối < 1e-7)', sai_lech < 1e-7, 'kiểm (P − Y)/n, chuyển vị, đạo hàm ReLU, phần phạt alpha·W/n')
kiem('cross_entropy ổn định với score rất lớn', np.isfinite(cross_entropy(np.array([[1000.0, 1001.0, 999.0] + [0.0] * 7]), np.array([1]))))

## Bước 3 · Huấn luyện mini-batch và checkpoint

Viết `huan_luyen(H, so_epoch=60, eta=1e-3, alpha=1e-4, B=256, seed=0)`: mỗi epoch xáo trộn, chia nhóm B ảnh, forward – backward – cập nhật (Adam như trang thuật toán, hoặc SGD nếu bạn muốn, ghi rõ). Cuối mỗi epoch ghi CE train, CE validation, accuracy validation; giữ **bản sao** tham số ở epoch có CE validation nhỏ nhất. Trả dict gồm `lich_su`, `P` (checkpoint), `epoch`, `ce_val`. Nếu checkpoint rơi đúng epoch cuối, mô hình có thể chưa học xong: ghi điều đó vào phần giới hạn, không âm thầm tăng epoch sau khi đã xem tập kiểm tra cuối.

In [ ]:
# TODO: def huan_luyen(...): ...  rồi  ket_qua = {64: huan_luyen(64)}
raise NotImplementedError('TODO bước 3: huan_luyen')
r64 = ket_qua[64]
for h in r64['lich_su'][::5]:
    print(f"epoch {h['epoch']:2d}: CE train {h['ce_train']:.3f} · CE val {h['ce_val']:.3f} · acc val {h['acc_val']:.3f}")
print('checkpoint: epoch', r64['epoch'])

In [ ]:
# TỰ KIỂM · bước 3
ls = r64['lich_su']
kiem('CE train giảm sau 60 epoch', ls[-1]['ce_train'] < ls[0]['ce_train'] - 0.2)
kiem('checkpoint đúng epoch có CE validation nhỏ nhất', r64['epoch'] == 1 + int(np.argmin([h['ce_val'] for h in ls])))
kiem('checkpoint là bản sao, không bị huấn luyện tiếp ghi đè', abs(cross_entropy(forward(r64['P'], V)['S'], yv) - r64['ce_val']) < 1e-12, 'dùng .copy() khi lưu')

## Bước 4 (khá) · Chọn cấu hình bằng validation; logistic đối chứng

**Quy tắc chọn (ghi trước khi xem kết quả):** huấn luyện thêm H = 16 với cùng mọi thứ khác; chọn H có **CE validation tại checkpoint nhỏ nhất**; nếu bằng nhau, lấy H nhỏ hơn. Fit logistic `LogisticRegression(C=1, solver='newton-cg', tol=1e-8, max_iter=1000)` trên cùng 8.000 ảnh để đối chứng (khoảng 15 giây).

In [ ]:
# TODO: ket_qua[16] = ...; H_chon theo quy tắc; P_chon, epoch_chon; logistic; du_doan_mlp(M) trả nhãn; acc_val = {'mlp': ..., 'logistic': ...}
raise NotImplementedError('TODO bước 4')
for h in sorted(ket_qua):
    print(f"H = {h}: checkpoint epoch {ket_qua[h]['epoch']}, CE val {ket_qua[h]['ce_val']:.4f}")
print('H chọn =', H_chon, '· accuracy validation:', acc_val)

In [ ]:
# TỰ KIỂM · bước 4
kiem('H chọn đúng quy tắc', H_chon == min(ket_qua, key=lambda h: (ket_qua[h]['ce_val'], h)))
kiem('dự đoán MLP dùng đúng checkpoint của H chọn', np.array_equal(du_doan_mlp(V[:50]), forward(ket_qua[H_chon]['P'], V[:50])['S'].argmax(1)))

## Bước 5 (khá) · Chấm tập kiểm tra cuối một lần, so cặp

Chỉ chạy ô này **sau khi** đã chốt cấu hình. `cham_cuoi` ghi lại lần chấm (chuỗi cấu hình và “dấu vân tay” của mô hình, tức mã băm dự đoán trên validation) vào `.project_bai10_cham_cuoi.json`; chạy lại notebook với **cùng** cấu hình và cùng mô hình thì nhận lại đúng kết quả cũ; đổi một trong hai sẽ bị chặn. Đây là cơ chế nhắc kỷ luật, không phải bảo mật; giảng viên đối chiếu tệp dấu vết khi chấm. Ô tự kiểm không cho biết đáp số của tập này.

In [ ]:
cau_hinh = f'H={H_chon}, epoch={epoch_chon}'
kq = cham_cuoi(cau_hinh, du_doan_mlp, logistic.predict)
se = lambda a: np.sqrt(a * (1 - a) / 1000)
print(f"MLP {kq['acc_mlp']:.3f} ± {se(kq['acc_mlp']):.3f} · logistic {kq['acc_logistic']:.3f} ± {se(kq['acc_logistic']):.3f}")
print(f"chỉ MLP đúng {kq['chi_mlp_dung']} · chỉ logistic đúng {kq['chi_logistic_dung']} · McNemar p = {kq['p_mcnemar']:.2g}")

In [ ]:
# TỰ KIỂM · bước 5
kiem('kết quả chấm cuối là các tỷ lệ hợp lệ', 0 <= kq['acc_mlp'] <= 1 and 0 <= kq['acc_logistic'] <= 1)
kiem('số ảnh bất đồng khớp hiệu accuracy', abs((kq['chi_mlp_dung'] - kq['chi_logistic_dung']) / 1000 - (kq['acc_mlp'] - kq['acc_logistic'])) < 1e-12)
try:
    cham_cuoi('H=999, epoch=1', du_doan_mlp, logistic.predict); chan = False
except RuntimeError:
    chan = True
kiem('hàm chặn chấm lần hai với cấu hình khác', chan)
try:
    cham_cuoi(cau_hinh, lambda M: np.zeros(len(M), int), logistic.predict); chan2 = False
except RuntimeError:
    chan2 = True
kiem('hàm chặn cả khi giữ chuỗi cấu hình nhưng đổi mô hình', chan2)

## Báo cáo một trang (nộp cùng notebook)

1. **Dữ liệu.** Ba tập, cách lấy, vì sao không dùng test chính thức.
2. **Mã.** Bằng chứng lan truyền ngược đúng (sai phân); loss ban đầu; đường loss train và validation.
3. **Chọn cấu hình.** Quy tắc đã ghi trước; bảng H – epoch checkpoint – CE validation – accuracy validation; cấu hình được chọn.
4. **Kết quả cuối.** Accuracy MLP và logistic trên tập kiểm tra cuối kèm sai số chuẩn; bảng bốn ô và McNemar; so với validation.
5. **Giới hạn.** Ít nhất ba: một seed và một lần chia; chỉ hai cấu hình; ảnh 28 × 28 đã căn giữa; không bất biến khi dịch ảnh (xem nâng cao B).

## Nâng cao (chọn một)

- **A · Đơn vị ẩn so với thành phần PCA.** Với 6 đơn vị có trọng số ra lớn nhất, lấy 100 ảnh fit làm đơn vị bật mạnh nhất, đo tỷ lệ của loại chiếm nhiều nhất (“độ thuần nhãn”); làm tương tự với 100 ảnh có score lớn nhất trên mỗi thành phần chính đầu. So sánh và giải thích bằng cách hai biểu diễn được học (theo nhãn / theo phương sai). Vẽ ảnh trung bình của 100 ảnh đó (như slide 57).
- **B · Ảnh dịch.** Đo accuracy validation khi dịch ảnh sang phải 1 và 2 pixel. Sau đó huấn luyện lại với tập fit được tăng cường (thêm bản dịch ngẫu nhiên −2…2 pixel), đo lại trên ảnh gốc và ảnh dịch. Báo cáo cả hai, không chỉ con số tốt.

In [ ]:
# Nâng cao A (tuỳ chọn, không có ô tự kiểm; chấm theo rubric): độ thuần nhãn của đơn vị ẩn và của thành phần PCA, ảnh trung bình

In [ ]:
# Nâng cao B (tuỳ chọn, không có ô tự kiểm; chấm theo rubric): đo trên ảnh dịch; huấn luyện với ảnh dịch, đo lại